# k-NN V1

For this first experiment, I used k-NN with k=3 and Euclidean distance. I started with 1000 training samples and 200 validation samples for each dataset to see how well it performs before trying different parameters.


First, I imported the libraries and functions I'll need.


In [16]:
from pathlib import Path
from time import perf_counter
import sys

import pandas as pd

# Allow imports when Jupyter starts in the repository root or notebooks folder.
project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_task, split_train_val
from src.models.knn import KNNClassifier


I started by testing the model on the image dataset.


In [17]:
# load_task also reads Kaggle test data, which V1 does not use.
X, y, _, _ = load_task("image")
X_train, X_val, y_train, y_val = split_train_val(
    X, y, val_fraction=0.2, seed=42
)

# Select only from the separate, already shuffled splits.
X_train_v1 = X_train[:1000]
y_train_v1 = y_train[:1000]
X_val_v1 = X_val[:200]
y_val_v1 = y_val[:200]

clf = KNNClassifier(k=3)
clf.fit(X_train_v1, y_train_v1)

start = perf_counter()
y_pred = clf.predict(X_val_v1)
prediction_time = perf_counter() - start
accuracy = sum(p == t for p, t in zip(y_pred, y_val_v1)) / len(y_val_v1)
image_accuracy = accuracy

print("Image V1: k=3, Euclidean distance, majority voting, no scaling")
print(f"Training samples: {len(X_train_v1)}")
print(f"Validation samples: {len(X_val_v1)}")
print(f"Validation accuracy: {accuracy:.2%}")
print(f"Prediction time: {prediction_time:.6f} seconds")


Image V1: k=3, Euclidean distance, majority voting, no scaling
Training samples: 1000
Validation samples: 200
Validation accuracy: 21.00%
Prediction time: 1.507990 seconds


I saved the predictions and accuracy so I can compare them with the next experiments.


In [18]:
result_dir = project_root / "results" / "image"
result_dir.mkdir(parents=True, exist_ok=True)

# These indices are positions in the shared validation split, not CSV row IDs.
validation_results = pd.DataFrame({
    "validation_index": range(len(y_val_v1)),
    "actual_label": y_val_v1,
    "predicted_label": y_pred,
})
validation_results.to_csv(result_dir / "knn_v1_validation.csv", index=False)

summary = pd.DataFrame([{
    "dataset": "image",
    "version": "V1",
    "k": clf.k,
    "distance_metric": "euclidean",
    "voting_method": "majority",
    "normalized": False,
    "training_samples": len(X_train_v1),
    "validation_samples": len(X_val_v1),
    "validation_accuracy": accuracy,
    "prediction_time_seconds": prediction_time,
}])
summary.to_csv(result_dir / "knn_v1_summary.csv", index=False)


Then I repeated the same test with the sentiment dataset.


In [ ]:
X, y, _, _ = load_task("sentiment")
X_train, X_val, y_train, y_val = split_train_val(
    X, y, val_fraction=0.2, seed=42
)

# Select only from the separate, already shuffled splits.
X_train_v1 = X_train[:1000]
y_train_v1 = y_train[:1000]
X_val_v1 = X_val[:200]
y_val_v1 = y_val[:200]

clf = KNNClassifier(k=3)
clf.fit(X_train_v1, y_train_v1)

start = perf_counter()
y_pred = clf.predict(X_val_v1)
prediction_time = perf_counter() - start
accuracy = sum(p == t for p, t in zip(y_pred, y_val_v1)) / len(y_val_v1)
sentiment_accuracy = accuracy

print("Sentiment V1: k=3, Euclidean distance, majority voting, no scaling")
print(f"Training samples: {len(X_train_v1)}")
print(f"Validation samples: {len(X_val_v1)}")
print(f"Validation accuracy: {accuracy:.2%}")
print(f"Prediction time: {prediction_time:.6f} seconds")


Sentiment V1: k=3, Euclidean distance, majority voting, no scaling
Training samples: 1000
Validation samples: 200
Validation accuracy: 57.50%
Prediction time: 1.500415 seconds


I saved the sentiment results as well.


In [20]:
result_dir = project_root / "results" / "sentiment"
result_dir.mkdir(parents=True, exist_ok=True)

# These indices are positions in the shared validation split, not CSV row IDs.
validation_results = pd.DataFrame({
    "validation_index": range(len(y_val_v1)),
    "actual_label": y_val_v1,
    "predicted_label": y_pred,
})
validation_results.to_csv(result_dir / "knn_v1_validation.csv", index=False)

summary = pd.DataFrame([{
    "dataset": "sentiment",
    "version": "V1",
    "k": clf.k,
    "distance_metric": "euclidean",
    "voting_method": "majority",
    "normalized": False,
    "training_samples": len(X_train_v1),
    "validation_samples": len(X_val_v1),
    "validation_accuracy": accuracy,
    "prediction_time_seconds": prediction_time,
}])
summary.to_csv(result_dir / "knn_v1_summary.csv", index=False)


Finally, I compared the accuracy for both datasets.


In [21]:
print("V1 validation accuracies (1,000 training / 200 validation samples each)")
print(f"Image:     {image_accuracy:.2%}")
print(f"Sentiment: {sentiment_accuracy:.2%}")


V1 validation accuracies (1,000 training / 200 validation samples each)
Image:     21.00%
Sentiment: 57.50%
